# 3 · Starter analysis

First indicators, to check the data and get an overview. **Scope** is decided once, in the next cells: by default, research publications (articles, reviews, conference papers, chapters, books, preprints, theses, reports) with at least one author signing with the institution.

Caveats worth keeping in mind: OpenAlex often holds a paper and its preprint or HAL deposit as two works — `deduplicate` keeps the published one; recent years are incomplete (indexing lag) and have had less time to be cited; FWCI compares citations to works of the same field, year and type.

In [ ]:
# Run this cell first. On Google Colab it installs the package; locally it does nothing
# (install once with `pip install -e .` from the project folder).
import sys

if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/celoy/bibliometry"
    !pip install -q "git+{REPO_URL}"

In [ ]:
from bibliometry.env import project_root

ROOT = project_root()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from bibliometry import analysis as A
from bibliometry.config import Config
from bibliometry.tables import read_tables

config = Config.load(ROOT / "config.yaml")
t = read_tables(config.tables_dir)
works_all, people = t["works"], t["people"]
works = A.in_scope(works_all)  # signed_only=False adds the people's works signed elsewhere
works = A.deduplicate(works)   # one row per title: drops preprint/HAL copies of the same paper
print(f"{len(works):,} works in scope out of {len(works_all):,} harvested "
      f"({(works.n_versions > 1).sum():,} had several versions)")

## Output over time

In [ ]:
fig = A.plot_lines(A.works_per_year(works), "Publications per year")

In [ ]:
per_lab = A.works_per_lab(works, people)
fig = A.plot_lines(per_lab, "Publications per year and lab (signed by people of the list)")
per_lab

In [ ]:
fig = A.plot_bars(A.type_counts(works), "Publication types")

## Coverage of the list of people

In [ ]:
A.roster_coverage(works, works_all)

## Venues, open access, citations

In [ ]:
A.top_venues(works[works.source_type != "repository"], 20)  # journals, conferences, books

In [ ]:
fig = A.plot_stacked_shares(A.open_access(works), "Open-access status (share of publications)")

In [ ]:
A.citation_summary(works).round(2)

## Collaborations and topics

In [ ]:
fig = A.plot_lines(A.international_share(works).to_frame(), "Share of internationally co-authored publications", ylabel="share")

In [ ]:
A.partners(t["authorships"], works, config.institution.id, 25)

In [ ]:
fig = A.plot_bars(A.partner_countries(t["authorships"], works, 20), "Countries of co-signing institutions")

In [ ]:
fig = A.plot_bars(A.top_topics(t["topics"], works, "field", 15), "OpenAlex fields (primary topic)")